In [ ]:
# 설치
!pip install transformers==4.45.0 -q
!pip install datasets -q
!pip install scikit-learn -q

In [ ]:
import shutil
shutil.rmtree("/root/.cache/huggingface", ignore_errors=True)

In [ ]:
import os
os.makedirs("/kaggle/working/distilbert", exist_ok=True)

base = "https://hf-mirror.com/distilbert-base-uncased/resolve/main"
for fn in ["model.safetensors", "config.json", "vocab.txt", "tokenizer_config.json"]:
    !wget -c {base}/{fn} -O /kaggle/working/distilbert/{fn}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DistilBertModel


# ─── 1. Dual-Stream Prototype Manifold ────────────────────────────────────────
class PrototypeManifold(nn.Module):
    def __init__(self, embed_dim: int, n_prototypes: int = 32, temperature: float = 0.1):
        super().__init__()
        self.temperature = temperature
        half = n_prototypes // 2
        self.proto_s = nn.Parameter(torch.randn(half, embed_dim))
        self.proto_c = nn.Parameter(torch.randn(half, embed_dim))

    def _stream_stats(self, z: torch.Tensor, z_norm: torch.Tensor, proto: torch.Tensor):
        p_norm  = F.normalize(proto, dim=-1)
        sim     = z_norm @ p_norm.T                                  # (B, K) cosine

        max_sim, _ = sim.max(dim=-1)                                 # (B,) → support/counter
        K          = proto.size(0)
        top_i      = sim.argmax(dim=-1)

        # ── ambiguity (tail): top-1 제거 후 나머지 분포 entropy ──────────
        mask     = F.one_hot(top_i, K).bool()
        sim_tail = sim.masked_fill(mask, float('-inf'))
        q_tail   = F.softmax(sim_tail / self.temperature, dim=-1)
        ent_tail = -(q_tail * (q_tail + 1e-8).log()).sum(dim=-1)
        max_ent  = torch.log(torch.tensor(float(K - 1), device=z.device))
        ambiguity_tail = (ent_tail / max_ent).clamp(0.0, 1.0)

        # ── ambiguity (gap): top1-top2 격차 (support 크기와 직교) ────────
        top2 = sim.topk(2, dim=-1).values                            # (B, 2)
        gap  = (top2[:, 0] - top2[:, 1]).clamp(min=0.0)
        ambiguity_gap = torch.exp(-gap / self.temperature).clamp(0.0, 1.0)

        return max_sim, ambiguity_tail, ambiguity_gap

    def forward(self, z: torch.Tensor) -> dict:
        z_norm = F.normalize(z, dim=-1)

        support, amb_tail_s, amb_gap_s = self._stream_stats(z, z_norm, self.proto_s)
        counter, amb_tail_c, amb_gap_c = self._stream_stats(z, z_norm, self.proto_c)

        # ambiguity: stream 내부 경쟁 (게이트 없음, stream-간 크기와 직교)
        ambiguity_score = torch.max(amb_gap_s, amb_gap_c).clamp(0.0, 1.0)   # field용 (gap)
        ambiguity_tail  = torch.max(amb_tail_s, amb_tail_c).clamp(0.0, 1.0) # probe용 (비교)

        # ── prototype collapse 방지 ──────────────────────────────────────
        s_norm = F.normalize(self.proto_s, dim=-1)
        c_norm = F.normalize(self.proto_c, dim=-1)

        cross_div = (s_norm @ c_norm.T).abs().mean()

        K_s     = s_norm.size(0)
        K_c     = c_norm.size(0)
        gram_s  = s_norm @ s_norm.T
        gram_c  = c_norm @ c_norm.T
        eye_s   = torch.eye(K_s, device=z.device)
        eye_c   = torch.eye(K_c, device=z.device)
        intra_s = (gram_s - eye_s).pow(2).mean()
        intra_c = (gram_c - eye_c).pow(2).mean()

        diversity_loss = cross_div + 0.5 * (intra_s + intra_c)

        return {
            "support":         support,
            "counter":         counter,
            "ambiguity_score": ambiguity_score,
            "ambiguity_tail":  ambiguity_tail,    # probe 비교용
            "diversity_loss":  diversity_loss,
        }

# ─── 2. Epistemic Field Classifier ────────────────────────────────────────────
class EpistemicFieldClassifier(nn.Module):

    AXES = ['truth', 'error', 'contradiction', 'novelty', 'ambiguity', 'ignorance']

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps        = eps
        self.scales     = nn.Parameter(torch.ones(6))
        self.truth_temp = nn.Parameter(torch.tensor(0.2))

        self.contra_bias = nn.Parameter(torch.tensor(1.0))  
        self.contra_temp = nn.Parameter(torch.tensor(0.2))

    def forward(self, manifold_out: dict, ignorance: torch.Tensor) -> dict:
        support         = manifold_out["support"]
        counter         = manifold_out["counter"]
        novelty_score   = manifold_out["novelty_score"]
        ambiguity_score = manifold_out["ambiguity_score"]

        temp     = F.softplus(self.truth_temp).clamp(min=0.05)
        margin_s = (support - counter).clamp(min=0.0)
        margin_c = (counter - support).clamp(min=0.0)

        # ── evidence plane ───────────────────────────────────────────────
        truth = support * torch.sigmoid(margin_s / temp)
        error = counter * torch.sigmoid(margin_c / temp)

        energy_sc     = support + counter
        agree         = (support - counter).abs().clamp(0.0, 1.0)
        c_temp        = F.softplus(self.contra_temp).clamp(min=0.05)
        contradiction = torch.sigmoid((energy_sc - F.softplus(self.contra_bias)) / c_temp) * (1.0 - agree)

        # ── independent uncertainty sources ──────────────────────────────
        novelty   = novelty_score
        ambiguity = ambiguity_score

        raw         = torch.stack(
            [truth, error, contradiction, novelty, ambiguity, ignorance], dim=-1
        )
        scales_norm = F.softmax(self.scales, dim=0) * 6.0
        field       = raw * scales_norm

        if self.training:
            dominant_type = None
            active_states = None
        else:
            dominant_idx  = field.argmax(dim=-1)
            dominant_type = [self.AXES[i] for i in dominant_idx.cpu().tolist()]
            active_states = []
            for sample in field:
                th     = sample.mean()
                active = [axis for idx, axis in enumerate(self.AXES) if sample[idx] > th]
                active_states.append(active if active else ["ignorance"])

        return {
            "field":         field,
            "dominant_type": dominant_type,
            "active_states": active_states,
            "truth":         truth,
            "error":         error,
            "contradiction": contradiction,
            "novelty":       novelty,
            "ambiguity":     ambiguity,
            "ignorance":     ignorance,
            "support_raw":   support,
            "counter_raw":   counter,
        }

# ─── Token-Novelty Source ─────────────────────────────────────────────────────
class TokenNovelty(nn.Module):
    """학습 어휘 대비 입력의 신규성 — 표면 형태, task/z와 완전 독립."""

    def __init__(self, vocab_size: int, eps: float = 1e-4):
        super().__init__()
        self.eps = eps
        self.register_buffer("token_count", torch.zeros(vocab_size))
        self.register_buffer("total", torch.tensor(0.0))
        # 특수토큰 마스킹용 (PAD/CLS/SEP은 신규성 계산서 제외)
        self.register_buffer("special_mask", torch.zeros(vocab_size, dtype=torch.bool))

    def set_special_tokens(self, ids):
        self.special_mask[torch.tensor(ids)] = True

    @torch.no_grad()
    def _update(self, input_ids, attention_mask):
        valid = input_ids[attention_mask.bool()]
        self.token_count.index_add_(0, valid, torch.ones_like(valid, dtype=torch.float))
        self.total += valid.numel()

    def forward(self, input_ids, attention_mask):
        if self.training:
            self._update(input_ids, attention_mask)

        # 토큰별 학습빈도 → 희귀도 = -log(freq), 미등장이면 최대
        total = self.total.clamp(min=1.0)
        freq  = self.token_count[input_ids] / total          # (B, T)
        rarity = -(freq + self.eps).log()                    # 희귀할수록 큼

        # 특수토큰·패딩 제외하고 문장 평균
        mask   = attention_mask.bool() & ~self.special_mask[input_ids]
        rarity = rarity * mask.float()
        denom  = mask.float().sum(dim=-1).clamp(min=1.0)
        sent_rarity = rarity.sum(dim=-1) / denom             # (B,)

        # EMA 없이 즉석 정규화 — log(total) 기준 (미등장 토큰 = -log(eps/total) 부근)
        max_rarity  = -torch.log(torch.tensor(self.eps, device=input_ids.device))
        novelty = (sent_rarity / max_rarity).clamp(0.0, 1.0)
        return novelty.detach()

# ─── Attention-based Ignorance Source ─────────────────────────────────────────
class AttentionIgnorance(nn.Module):
    """입력의 정보 결핍 — attention dispersion. classifier/manifold와 독립 소스."""

    def __init__(self, momentum: float = 0.01, eps: float = 1e-4):
        super().__init__()
        self.eps = eps
        self.register_buffer("ent_mean", torch.tensor(0.5))   # 정규화용 EMA
        self.register_buffer("ent_std",  torch.tensor(0.15))
        self.register_buffer("initialized", torch.tensor(False))
        self.momentum = momentum

    @torch.no_grad()
    def _update(self, e):
        m = self.momentum
        if not self.initialized:
            self.ent_mean.copy_(e.mean())
            self.ent_std.copy_(e.std() + self.eps)
            self.initialized.fill_(True)
        else:
            self.ent_mean.mul_(1 - m).add_(e.mean(), alpha=m)
            self.ent_std.mul_(1 - m).add_(e.std() + self.eps, alpha=m)

    def forward(self, attentions, attention_mask):
        # attentions: tuple of (B, H, T, T), 마지막 layer 사용
        attn = attentions[-1]                              # (B, H, T, T)
        # 모든 query 토큰의 attention 분포 entropy, head·token 평균 (CLS만 아님)
        mask = attention_mask.unsqueeze(1).unsqueeze(1)    # (B,1,1,T)
        a    = attn * mask
        a    = a / (a.sum(dim=-1, keepdim=True) + self.eps)
        ent  = -(a * (a + self.eps).log()).sum(dim=-1)     # (B, H, T) 각 query의 entropy
        # query 쪽도 padding 제외하고 평균
        qmask   = attention_mask.unsqueeze(1).float()      # (B,1,T)
        ent_tok = (ent.mean(dim=1) * qmask.squeeze(1))     # (B, T) head 평균
        lengths = attention_mask.sum(dim=-1).float()       # (B,)
        ent_mean_per = ent_tok.sum(dim=-1) / (lengths + self.eps)  # (B,) 토큰 평균

        # 길이로 정규화 (긴 시퀀스 entropy 상한 보정)
        ent_norm = ent_mean_per / (lengths.log() + self.eps)

        if self.training:
            self._update(ent_norm.detach())

        # EMA 기준 표준화 → sigmoid로 [0,1]
        mu  = self.ent_mean.detach()
        std = self.ent_std.detach().clamp(min=self.eps)
        ignorance = torch.sigmoid((ent_norm - mu) / std)
        return ignorance.detach()
        
# ─── 3. EpistemicBERT ─────────────────────────────────────────────────────────
class EpistemicBERT(nn.Module):

    def __init__(
        self,
        n_classes:    int  = 3,
        n_prototypes: int  = 32,
        proj_dim:     int  = 128,
        freeze_bert:  bool = False,
    ):
        super().__init__()

        self.bert      = DistilBertModel.from_pretrained("/kaggle/working/distilbert")
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.proj      = nn.Linear(768, proj_dim)
        self.manifold  = PrototypeManifold(proj_dim, n_prototypes)
        self.token_nov = TokenNovelty(self.bert.config.vocab_size)
        self.attn_ign  = AttentionIgnorance() 
        self.epistemic = EpistemicFieldClassifier()

        self.field_proj = nn.Linear(6, n_classes)
        self.z_proj     = nn.Linear(proj_dim, n_classes)

        self.margin_param     = nn.Parameter(torch.tensor(0.2))
        self.energy_ceiling   = nn.Parameter(torch.tensor(0.5))
        self.con_energy_floor = nn.Parameter(torch.tensor(0.5))

    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(
            input_ids=input_ids, attention_mask=attention_mask,
            output_attentions=True,                            # attention 활성화
        )
        cls = bert_out.last_hidden_state[:, 0]

        z             = self.proj(cls)
        manifold_out  = self.manifold(z)
        manifold_out["novelty_score"] = self.token_nov(input_ids, attention_mask)

        # ── ignorance = attention dispersion (정보 결핍, 별도 소스) ────────
        ignorance = self.attn_ign(bert_out.attentions, attention_mask)

        epistemic_out = self.epistemic(manifold_out, ignorance)

        field  = epistemic_out["field"]
        logits = self.field_proj(field) + self.z_proj(z)

        return logits, epistemic_out, manifold_out["diversity_loss"]

import torch
import torch.nn.functional as F
import numpy as np
from sklearn.linear_model import LinearRegression


@torch.no_grad()
def identifiability_probe(model, loader, device):
    model.eval()
    AXES = ['truth', 'error', 'contradiction', 'novelty', 'ambiguity', 'ignorance']

    fields = []
    for batch in loader:
        _, eout, _ = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        fields.append(eout["field"].cpu().float())
    F_mat = torch.cat(fields, dim=0).numpy()

    from sklearn.linear_model import LinearRegression
    import numpy as np

    # plane 축(0,1,2)과 uncertainty 축(3,4,5) 그룹 구분해서 해석
    print(f"\n{'axis':>16}  {'R² from others':>16}  {'group':>10}  {'verdict':>14}")
    print("─" * 62)
    groups = {0:"plane",1:"plane",2:"plane",3:"uncert",4:"uncert",5:"uncert"}
    for i, ax in enumerate(AXES):
        y   = F_mat[:, i]
        X   = np.delete(F_mat, i, axis=1)
        r2  = LinearRegression().fit(X, y).score(X, y)
        # plane 축은 redundant가 정상(2D 좌표계), uncert 축만 independent 기대
        if groups[i] == "plane":
            verdict = "plane-coord (OK)" if r2 > 0.6 else "unexpected-indep"
        else:
            verdict = "LEAK" if r2 > 0.6 else "independent (OK)"
        print(f"{ax:>16}  {r2:>16.4f}  {groups[i]:>10}  {verdict:>14}")

    corr = np.corrcoef(F_mat.T)
    print(f"\n  |correlation| matrix:")
    print(f"{'':>14}" + "".join(f"{a[:5]:>8}" for a in AXES))
    for i, ax in enumerate(AXES):
        row = "".join(f"{abs(corr[i, j]):>8.3f}" for j in range(6))
        print(f"{ax:>14}{row}")

import torch
import torch.nn.functional as F

# ─── Novelty / Ignorance Disentanglement Probe ────────────────────────────────
# 각 그룹은 (premise, hypothesis) — SNLI 입력 형식 유지
PROBE = {
    # G1: 정보 충분 + 문장 명확 + 구조가 낯섦 → novelty↑, ignorance↓
    "G1_novel_informed": [
        ("The quantum self-referential reasoner collapsed its own eigenstate.",
         "A self-modeling quantum device altered the particle it measured."),
        ("Zeta-7, discovered in 2029, decays into mirror-charged leptons.",
         "The Zeta-7 particle produces leptons with inverted charge."),
        ("The neuromorphic compiler hallucinated a non-Euclidean memory lattice.",
         "A brain-like compiler generated an impossible memory structure."),
        ("Synthetic ribozymes folded into a topology unseen in nature.",
         "Artificial RNA enzymes adopted a novel three-dimensional shape."),
        ("The exo-linguistic glyphs encoded a base-12 recursive grammar.",
         "The alien symbols followed a recursive twelve-base language system."),
    ],
    # G2: 정보 부족 + underspecified → ignorance↑, novelty↓
    "G2_ignorant_vague": [
        ("Something happened somewhere to someone.",
         "It probably turned out a certain way."),
        ("A particle was detected at some point.",
         "The thing was maybe important."),
        ("He said it might be the case, perhaps.",
         "The situation could possibly be relevant."),
        ("The truth about the event remains unknown.",
         "Nobody can say what really occurred."),
        ("They did the thing in the place that time.",
         "It was somehow related to the matter."),
    ],
    # G3: 평범한 익숙한 입력 → 둘 다 낮음 (기준선)
    "G3_ordinary": [
        ("A man is walking his dog in the park.",
         "A person is outside with an animal."),
        ("The woman bought groceries at the store.",
         "Someone purchased food at a shop."),
        ("Children are playing soccer on the field.",
         "Kids are playing a sport outdoors."),
        ("A chef is cooking pasta in the kitchen.",
         "A person is preparing food."),
        ("The train arrived at the station on time.",
         "A train reached its stop as scheduled."),
    ],
}


@torch.no_grad()
def novelty_ignorance_probe(model, tokenizer, device, max_length=128):
    model.eval()
    AXES = ['truth', 'error', 'contradiction', 'novelty', 'ambiguity', 'ignorance']

    print(f"\n{'group':>20}  {'novelty':>9}  {'ignorance':>10}  {'nov-ign gap':>12}")
    print("─" * 58)

    group_means = {}
    for gname, pairs in PROBE.items():
        prem = [p for p, h in pairs]
        hyp  = [h for p, h in pairs]
        enc  = tokenizer(
            prem, hyp, max_length=max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        _, eout, _ = model(
            enc["input_ids"].to(device), enc["attention_mask"].to(device)
        )
        nov = eout["novelty"].mean().item()
        ign = eout["ignorance"].mean().item()
        group_means[gname] = {ax: eout[ax].mean().item() for ax in AXES}
        print(f"{gname:>20}  {nov:>9.4f}  {ign:>10.4f}  {nov - ign:>+12.4f}")

    # 핵심 판정: G1과 G2에서 두 축이 반대로 갈리는가
    g1, g2 = group_means["G1_novel_informed"], group_means["G2_ignorant_vague"]
    nov_sep = g1["novelty"]   - g2["novelty"]      # >0 이어야: G1이 더 novel
    ign_sep = g2["ignorance"] - g1["ignorance"]    # >0 이어야: G2가 더 ignorant

    print(f"\n  ── separation test ──")
    print(f"  novelty(G1) - novelty(G2)     = {nov_sep:>+.4f}   (>0 기대: G1이 더 낯섦)")
    print(f"  ignorance(G2) - ignorance(G1) = {ign_sep:>+.4f}   (>0 기대: G2가 정보부족)")

    if nov_sep > 0.03 and ign_sep > 0.03:
        print(f"  → 두 축이 의도대로 분리됨. nov↔ign 상관은 SNLI 데이터 아티팩트.")
    elif nov_sep > 0.03 or ign_sep > 0.03:
        print(f"  → 부분 분리. 한 축은 작동, 다른 축은 약함.")
    else:
        print(f"  → 분리 실패. 두 축이 같은 것을 측정 (측정 중복 가능성).")

    # 전체 field 프로파일 (각 그룹이 어느 축을 켜는지)
    print(f"\n  ── full field profile ──")
    print(f"{'group':>20}" + "".join(f"{a[:6]:>9}" for a in AXES))
    for gname, vals in group_means.items():
        print(f"{gname:>20}" + "".join(f"{vals[a]:>9.4f}" for a in AXES))

    return group_means

@torch.no_grad()
def attention_entropy_probe(model, tokenizer, device, max_length=128):
    model.eval()
    # DistilBERT attention 활성화
    model.bert.config.output_attentions = True

    print(f"\n{'group':>20}  {'attn_entropy':>13}  {'pred_entropy(ign)':>18}")
    print("─" * 56)

    for gname, pairs in PROBE.items():
        prem = [p for p, h in pairs]
        hyp  = [h for p, h in pairs]
        enc  = tokenizer(prem, hyp, max_length=max_length, padding="max_length",
                         truncation=True, return_tensors="pt")
        ids  = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)

        out  = model.bert(input_ids=ids, attention_mask=mask)
        # 마지막 layer, [CLS]가 각 토큰에 주는 attention, head 평균
        attn = out.attentions[-1]                       # (B, H, T, T)
        cls_attn = attn[:, :, 0, :].mean(dim=1)         # (B, T)  CLS→tokens, head mean
        # padding 마스킹 후 정규화
        cls_attn = cls_attn * mask
        cls_attn = cls_attn / (cls_attn.sum(dim=-1, keepdim=True) + 1e-8)
        ent      = -(cls_attn * (cls_attn + 1e-8).log()).sum(dim=-1)
        # 길이 정규화 (긴 문장이 자동으로 entropy 높으니)
        lengths  = mask.sum(dim=-1).float()
        ent_norm = ent / (lengths.log() + 1e-8)

        # 비교용 pred entropy (현재 ignorance)
        _, eout, _ = model(ids, mask)
        ign = eout["ignorance"].mean().item()

        print(f"{gname:>20}  {ent_norm.mean().item():>13.4f}  {ign:>18.4f}")

    model.bert.config.output_attentions = False

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizerFast
from datasets import load_dataset
from tqdm import tqdm
from collections import defaultdict
import json

# ─── Config ───────────────────────────────────────────────────────────────────
CFG = dict(
    model = dict(
        n_classes    = 3,
        n_prototypes = 32,
        proj_dim     = 128,
        freeze_bert  = False,
    ),
    train = dict(
        batch_size        = 64,
        epochs            = 3,
        lr_bert           = 2e-5,
        lr_head           = 1e-3,
        max_length        = 128,
        lambda_ce         = 0.3,    # CE: secondary
        lambda_field      = 1.0,    # ranking loss: primary
        lambda_margin     = 0.2,    # stream margin loss
        lambda_diversity  = 0.1,    # proto stream diversity
        ranking_margin    = 0.1,    # field ranking margin
        train_size        = 50_000,
        val_size          = 5_000,
        lambda_disent = 0.03,
    ),
    device = "cuda" if torch.cuda.is_available() else "cpu",
    seed   = 42,
)


# ─── Dataset ──────────────────────────────────────────────────────────────────
class NLIDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_length):
        self.data       = hf_split.filter(lambda x: x["label"] != -1)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item["premise"], item["hypothesis"],
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(item["label"], dtype=torch.long),
        }


class OODDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_length):
        self.data       = hf_split
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item["sentence1"], item["sentence2"],
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


# ─── Loss ─────────────────────────────────────────────────────────────────────
def field_ranking_loss(field, labels, margin: float = 0.1, eps: float = 1e-8):

    loss     = torch.tensor(0.0, device=field.device)
    n        = torch.tensor(0,   device=field.device)

    ent_mask = (labels == 0)
    neu_mask = (labels == 1)
    con_mask = (labels == 2)

    if ent_mask.any():
        f = field[ent_mask]
        for idx in [1, 2, 3, 4, 5]:
            loss += F.relu(margin - (f[:, 0] - f[:, idx])).mean()
        n += ent_mask.sum()

    if neu_mask.any():
        f = field[neu_mask]
        for idx in [0, 1, 2, 3, 5]:
            loss += F.relu(margin - (f[:, 4] - f[:, idx])).mean()
        n += neu_mask.sum()

    if con_mask.any():
        f = field[con_mask]
        for idx in [0, 1, 3, 4, 5]:
            loss += F.relu(margin - (f[:, 2] - f[:, idx])).mean()
        # truth ≈ error: 두 stream 모두 강한 고에너지 상태
        loss += (f[:, 0] - f[:, 1]).abs().mean()
        n += con_mask.sum()

    return loss / (n.float() + eps)


def margin_loss(support, counter, labels, margin_param, energy_ceiling, con_energy_floor):
    margin  = F.softplus(margin_param).clamp(min=0.05)
    ceiling = F.softplus(energy_ceiling).clamp(min=0.2)
    e_floor = F.softplus(con_energy_floor).clamp(min=0.2)

    loss     = torch.tensor(0.0, device=support.device)
    ent_mask = (labels == 0)
    neu_mask = (labels == 1)
    con_mask = (labels == 2)

    if ent_mask.any():
        loss += F.relu(counter[ent_mask] - support[ent_mask] + margin).mean()

    if neu_mask.any():
        energy  = support[neu_mask] ** 2 + counter[neu_mask] ** 2
        balance = (support[neu_mask] - counter[neu_mask]).abs()
        loss   += F.relu(energy - ceiling).mean() + balance.mean()

    if con_mask.any():
        gap    = (support[con_mask] - counter[con_mask]).abs()
        energy = support[con_mask] ** 2 + counter[con_mask] ** 2
        loss  += F.relu(gap - margin).mean()
        loss  += F.relu(e_floor - energy).mean()

    return loss / 3.0


# ─── Optimizer ────────────────────────────────────────────────────────────────
def build_optimizer(model, cfg):
    bert_params = list(model.bert.parameters())
    head_params = (
        list(model.proj.parameters())
        + list(model.manifold.parameters())
        + list(model.epistemic.parameters())
        + list(model.field_proj.parameters())
        + list(model.z_proj.parameters())
        + [model.margin_param, model.energy_ceiling, model.con_energy_floor]
    )
    return torch.optim.AdamW([
        {"params": bert_params, "lr": cfg["lr_bert"]},
        {"params": head_params, "lr": cfg["lr_head"]},
    ], weight_decay=1e-2)


# ─── Train ────────────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scaler, device, tcfg):
    
    def disentangle_loss(eout):
        # 독립이어야 하는 모든 uncert 쌍 + uncert-plane 쌍
        def corr(a, b):
            a_c = a - a.mean()
            b_c = b - b.mean()
            return ((a_c * b_c).mean() / (a_c.std() * b_c.std() + 1e-8)).abs()
    
        nov = eout["novelty"]
        amb = eout["ambiguity"]
        ign = eout["ignorance"]
        con = eout["contradiction"]
    
        loss = (
            corr(nov, amb)
            + corr(amb, ign)
        )
        return loss / 2.0
    
    model.train()
    ce_fn = nn.CrossEntropyLoss()
    total_loss = total_ce = total_field = total_correct = total = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits, eout, div_loss = model(input_ids, attention_mask)

            ce     = ce_fn(logits, labels)
            f_loss = field_ranking_loss(eout["field"], labels, tcfg["ranking_margin"])
            m_loss = margin_loss(
                eout["support_raw"], eout["counter_raw"], labels,
                model.margin_param, model.energy_ceiling, model.con_energy_floor,
            )
            d_loss = disentangle_loss(eout)
            loss = (
                tcfg["lambda_ce"]        * ce
                + tcfg["lambda_field"]   * f_loss
                + tcfg["lambda_margin"]  * m_loss
                + tcfg["lambda_diversity"] * div_loss
                + tcfg["lambda_disent"]    * d_loss
            )

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  nan/inf  ce={ce.item():.4f}  f={f_loss.item():.4f}  m={m_loss.item():.4f}")
            break

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total         += labels.size(0)
        total_loss    += loss.item()
        total_ce      += ce.item()
        total_field   += f_loss.item()

    n = len(loader)
    return {
        "loss":  total_loss  / n,
        "ce":    total_ce    / n,
        "field": total_field / n,
        "acc":   total_correct / total,
    }


# ─── Evaluate ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ce_fn = nn.CrossEntropyLoss()
    AXES  = EpistemicFieldClassifier.AXES

    axis_sums   = defaultdict(lambda: defaultdict(float))
    axis_counts = defaultdict(int)
    s_sums      = defaultdict(float)
    c_sums      = defaultdict(float)
    total_loss = total_correct = total = 0
    field_all  = []

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        with torch.cuda.amp.autocast():
            logits, eout, _ = model(input_ids, attention_mask)
            loss = ce_fn(logits, labels)

        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_loss    += loss.item()
        total         += labels.size(0)
        field_all.append(eout["field"].cpu().float())

        for b in range(labels.size(0)):
            lbl = labels[b].item()
            axis_counts[lbl] += 1
            s_sums[lbl] += eout["support_raw"][b].item()
            c_sums[lbl] += eout["counter_raw"][b].item()
            for a, ax in enumerate(AXES):
                axis_sums[lbl][ax] += eout["field"][b, a].item()

    label_names = {0: "entailment", 1: "neutral", 2: "contradiction"}
    field_by_label = {
        label_names[lbl]: {ax: axis_sums[lbl][ax] / axis_counts[lbl] for ax in AXES}
        for lbl in label_names if axis_counts[lbl] > 0
    }
    field_cat = torch.cat(field_all, dim=0)
    monitor   = {
        "field_mean": field_cat.mean(0).numpy().round(4).tolist(),
        "field_std":  field_cat.std(0).numpy().round(4).tolist(),
    }
    return {
        "loss":           total_loss / len(loader),
        "acc":            total_correct / total,
        "field_by_label": field_by_label,
        "monitor":        monitor,
        "support_by_label": {
            label_names[lbl]: {
                "support": s_sums[lbl] / axis_counts[lbl],
                "counter": c_sums[lbl] / axis_counts[lbl],
            }
            for lbl in label_names if axis_counts[lbl] > 0
        },
    }


@torch.no_grad()
def evaluate_ood(model, ood_loader, id_loader, device):
    model.eval()
    AXES = EpistemicFieldClassifier.AXES

    def collect(loader):
        fs = []
        for batch in loader:
            with torch.cuda.amp.autocast():
                _, eout, _ = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                )
            fs.append(eout["field"].cpu())
        return torch.cat(fs, dim=0)

    id_f  = collect(id_loader)
    ood_f = collect(ood_loader)
    return {
        ax: {"id_mean": id_f[:, i].mean().item(), "ood_mean": ood_f[:, i].mean().item()}
        for i, ax in enumerate(AXES)
    }


@torch.no_grad()
def evaluate_calibration(model, loader, device):
    model.eval()
    AXES = EpistemicFieldClassifier.AXES
    correct_fields, wrong_fields = [], []

    for batch in loader:
        with torch.cuda.amp.autocast():
            logits, eout, _ = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
            )
        field  = eout["field"].cpu()
        preds  = logits.argmax(dim=-1).cpu()
        labels = batch["label"].cpu()
        for b in range(labels.size(0)):
            (correct_fields if preds[b] == labels[b] else wrong_fields).append(field[b])

    cf = torch.stack(correct_fields) if correct_fields else torch.zeros(1, 6)
    wf = torch.stack(wrong_fields)   if wrong_fields   else torch.zeros(1, 6)
    return {
        ax: {"correct_mean": cf[:, i].mean().item(), "wrong_mean": wf[:, i].mean().item()}
        for i, ax in enumerate(AXES)
    }


# ─── Pretty Print ─────────────────────────────────────────────────────────────
def print_field_by_label(d):
    AXES = EpistemicFieldClassifier.AXES
    print(f"{'':>15}" + "".join(f"{a:>14}" for a in AXES))
    print("─" * (15 + 14 * len(AXES)))
    for name, vals in d.items():
        print(f"{name:>15}" + "".join(f"{vals[a]:>14.4f}" for a in AXES))

def print_ood_comparison(r):
    AXES = EpistemicFieldClassifier.AXES
    print(f"\n{'axis':>16}  {'ID mean':>10}  {'OOD mean':>10}  {'OOD - ID':>10}")
    print("─" * 52)
    for ax in AXES:
        id_m, ood_m = r[ax]["id_mean"], r[ax]["ood_mean"]
        flag = "  ← ↑" if ax in ("novelty", "ignorance", "ambiguity") and (ood_m - id_m) > 0.05 else ""
        print(f"{ax:>16}  {id_m:>10.4f}  {ood_m:>10.4f}  {ood_m - id_m:>+10.4f}{flag}")

def print_calibration(r):
    AXES = EpistemicFieldClassifier.AXES
    print(f"\n{'axis':>16}  {'correct':>10}  {'wrong':>10}  {'wrong - correct':>16}")
    print("─" * 58)
    for ax in AXES:
        c, w = r[ax]["correct_mean"], r[ax]["wrong_mean"]
        flag = "  ← ↑" if ax in ("error", "ignorance", "ambiguity", "novelty") and (w - c) > 0.03 else ""
        print(f"{ax:>16}  {c:>10.4f}  {w:>10.4f}  {w - c:>+16.4f}{flag}")


# ─── Main ─────────────────────────────────────────────────────────────────────
from datasets import concatenate_datasets
def main():
    torch.manual_seed(CFG["seed"])
    device = CFG["device"]
    tcfg   = CFG["train"]
    print(f"Device: {device}")

    tokenizer      = DistilBertTokenizerFast.from_pretrained("/kaggle/working/distilbert")
    
    print("Loading ANLI...")
    anli = load_dataset("anli")

    train_full = concatenate_datasets([anli["train_r1"], anli["train_r2"], anli["train_r3"]])
    val_full   = concatenate_datasets([anli["dev_r1"],   anli["dev_r2"],   anli["dev_r3"]])

    train_split = (
        train_full.select(range(min(tcfg["train_size"], len(train_full))))
        if tcfg["train_size"]
        else train_full
    )
    val_split = (
        val_full.select(range(min(tcfg["val_size"], len(val_full))))
        if tcfg["val_size"]
        else val_full
    )

    train_ds     = NLIDataset(train_split, tokenizer, tcfg["max_length"])
    val_ds       = NLIDataset(val_split,   tokenizer, tcfg["max_length"])
    train_loader = DataLoader(train_ds, batch_size=tcfg["batch_size"], shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=tcfg["batch_size"], shuffle=False, num_workers=0, pin_memory=True)

    print("Loading RTE (OOD)...")
    rte        = load_dataset("glue", "rte")
    ood_ds     = OODDataset(rte["validation"], tokenizer, tcfg["max_length"])
    ood_loader = DataLoader(ood_ds, batch_size=tcfg["batch_size"], shuffle=False, num_workers=0, pin_memory=True)


    model     = EpistemicBERT(**CFG["model"]).to(device)
    model.token_nov.set_special_tokens(tokenizer.all_special_ids)

    print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    optimizer = build_optimizer(model, tcfg)
    scaler    = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

    history = []
    for epoch in range(tcfg["epochs"]):
        print(f"\n═══ Epoch {epoch + 1}/{tcfg['epochs']} ═══")

        tr = train_epoch(model, train_loader, optimizer, scaler, device, tcfg)
        print(f"  train  loss={tr['loss']:.4f}  ce={tr['ce']:.4f}  field={tr['field']:.4f}  acc={tr['acc']:.4f}")
        print(f"  learnable params  margin={F.softplus(model.margin_param).item():.4f}  "
              f"energy_ceil={F.softplus(model.energy_ceiling).item():.4f}  "
              f"con_floor={F.softplus(model.con_energy_floor).item():.4f}  "
              f"truth_temp={F.softplus(model.epistemic.truth_temp).item():.4f}")

        vl = evaluate(model, val_loader, device)
        m  = vl["monitor"]
        print(f"  monitor  field_mean={m['field_mean']}")
        print(f"           field_std ={m['field_std']}")
        print(f"  val    loss={vl['loss']:.4f}  acc={vl['acc']:.4f}")

        print("\n  Epistemic field by label (val):")
        print_field_by_label(vl["field_by_label"])
        print("\n  Support / Counter by label:")
        for name, vals in vl["support_by_label"].items():
            print(f"  {name:>15}  support={vals['support']:.4f}  counter={vals['counter']:.4f}")

        history.append({"epoch": epoch + 1, "train": tr, "val": vl})

    print("\n\n═══ OOD Experiment ═══")
    print_ood_comparison(evaluate_ood(model, ood_loader, val_loader, device))

    print("\n\n═══ Calibration ═══")
    print_calibration(evaluate_calibration(model, val_loader, device))

    print("\n\n═══ Identifiability Probe ═══")
    identifiability_probe(model, val_loader, device)

    print("\n\n═══ Novelty/Ignorance Disentanglement Probe ═══")
    novelty_ignorance_probe(model, tokenizer, device)

    print("\n\n═══ Attention Entropy Probe (ignorance source 후보) ═══")
    attention_entropy_probe(model, tokenizer, device)

    torch.save(model.state_dict(), "/kaggle/working/epistemic_bert.pt")
    with open("/kaggle/working/results.json", "w") as f:
        json.dump({"history": history}, f, indent=2)


if __name__ == "__main__":
    main()